# Phase 2: Legal NER: GQA decoder

This is the **Legal NER** part of the full `gqa` notebook (`notebooks/CS781_gqa_all_phases.ipynb`), extracted so it can be read on its own. It reuses the tokenizer, model classes and helpers defined in the earlier cells, so **to run it, execute the full notebook from the top** (or run the Phase 1 set-up cells first, then this part).

## Fine Tuning

In [ ]:
import collections

def strict_scores(gold, pred, labels):
    """
    gold, pred: {doc_id: [[start, end, label, (text)], ...]}
    Exact match on (doc_id, start, end, label). Macro over `labels`.
    Returns dict with macro_p / macro_r / macro_f1, per-label dict, and micro_f1.
    """
    def to_set(d):
        out = set()
        for did, ents in d.items():
            for e in ents:
                out.add((str(did), int(e[0]), int(e[1]), e[2]))
        return out

    G, P = to_set(gold), to_set(pred)
    tp, gp, pp = collections.Counter(), collections.Counter(), collections.Counter()
    for x in G: gp[x[3]] += 1
    for x in P: pp[x[3]] += 1
    for x in G & P: tp[x[3]] += 1

    per = {}
    for lab in labels:
        p = tp[lab] / pp[lab] if pp[lab] else 0.0
        r = tp[lab] / gp[lab] if gp[lab] else 0.0
        f = 2 * p * r / (p + r) if (p + r) else 0.0
        per[lab] = {"p": p, "r": r, "f1": f, "support": gp[lab], "pred": pp[lab], "tp": tp[lab]}

    n = len(labels)
    macro_p = sum(v["p"] for v in per.values()) / n
    macro_r = sum(v["r"] for v in per.values()) / n
    macro_f1 = sum(v["f1"] for v in per.values()) / n
    TP, PP, GG = sum(tp.values()), sum(pp.values()), sum(gp.values())
    mp, mr = (TP / PP if PP else 0.0), (TP / GG if GG else 0.0)
    micro_f1 = 2 * mp * mr / (mp + mr) if (mp + mr) else 0.0
    return {"macro_p": macro_p, "macro_r": macro_r, "macro_f1": macro_f1,
            "micro_f1": micro_f1, "per_label": per}

def print_scores(res):
    print(f"MACRO  P={res['macro_p']:.4f}  R={res['macro_r']:.4f}  F1={res['macro_f1']:.4f}  | micro F1={res['micro_f1']:.4f}")
    print(f"{'label':10s} {'P':>7s} {'R':>7s} {'F1':>7s} {'gold':>6s} {'pred':>6s} {'tp':>6s}")
    for lab, v in sorted(res["per_label"].items(), key=lambda kv: -kv[1]["support"]):
        print(f"{lab:10s} {v['p']:7.3f} {v['r']:7.3f} {v['f1']:7.3f} {v['support']:6d} {v['pred']:6d} {v['tp']:6d}")

# ---------- self-test on toy data ----------
_L = ["JUDGE", "COURT", "DATE"]
_gold = {"d1": [[0, 5, "JUDGE", "Smith"], [10, 20, "COURT", "High Court"]],
         "d2": [[3, 9, "DATE", "1 May"]]}

# perfect predictions (with and without optional text field)
_r = strict_scores(_gold, {"d1": [[0, 5, "JUDGE"], [10, 20, "COURT", "x"]], "d2": [[3, 9, "DATE"]]}, _L)
assert abs(_r["macro_f1"] - 1.0) < 1e-9, _r

# boundary off by one and wrong label must both count as wrong
_r = strict_scores(_gold, {"d1": [[0, 6, "JUDGE"], [10, 20, "JUDGE"]], "d2": [[3, 9, "DATE"]]}, _L)
assert _r["per_label"]["JUDGE"]["tp"] == 0 and _r["per_label"]["COURT"]["tp"] == 0
assert _r["per_label"]["DATE"]["f1"] == 1.0
assert abs(_r["macro_f1"] - 1/3) < 1e-9, _r["macro_f1"]

# empty predictions -> zeros; duplicate predictions must not inflate counts
assert strict_scores(_gold, {}, _L)["macro_f1"] == 0.0
_r = strict_scores(_gold, {"d2": [[3, 9, "DATE"], [3, 9, "DATE"]]}, _L)
assert _r["per_label"]["DATE"]["pred"] == 1 and _r["per_label"]["DATE"]["p"] == 1.0
print("✅ strict scorer self-test passed")

✅ strict scorer self-test passed


In [ ]:
import os, json, math, time, random, collections
import numpy as np
import torch, torch.nn as nn, torch.nn.functional as F
from torch.optim.lr_scheduler import LambdaLR
from huggingface_hub import hf_hub_download
from tqdm.auto import tqdm

for name in ["tokenizer", "device", "HF_DATASET_REPO", "HF_TOKEN", "strict_scores", "print_scores"]:
    assert name in globals(), f"'{name}' is not defined — run the earlier cell that defines it."

# ============================================================
# 0. Config
# ============================================================
WINDOW, STRIDE = 1024, 768            # tokens per window / step (256 overlap)
VAL_FRACTION, SEED = 0.15, 42
USE_CLASS_WEIGHTS, CLASS_W_CLIP = True, 10.0

P2_DIR = "/content/drive/MyDrive/CS781_Capstone_Project_Phase_2"
if not os.path.isdir("/content/drive/MyDrive"):
    print("⚠️ Drive not mounted — saving Phase-2 files locally (lost if the runtime disconnects).")
    P2_DIR = "CS781_Capstone_Project_Phase_2"
os.makedirs(P2_DIR, exist_ok=True)

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
use_bf16 = torch.cuda.is_available() and torch.cuda.is_bf16_supported()
AMP_DTYPE = torch.bfloat16 if use_bf16 else torch.float16
PAD_ID = tokenizer.token_to_id("[PAD]")
PAD_ID = 1 if PAD_ID is None else PAD_ID
def amp(): return torch.amp.autocast(device_type="cuda", dtype=AMP_DTYPE)

# ============================================================
# 1. Load Phase-2 files
# ============================================================
BASE = "Phase_2_LNER"
def _path(fn): return hf_hub_download(repo_id=HF_DATASET_REPO, filename=f"{BASE}/{fn}", repo_type="dataset", token=HF_TOKEN)
def _json(fn):
    with open(_path(fn), "r", encoding="utf-8") as f: return json.load(f)

text1, gold1 = _json("Train/fold1-text.json"), _json("Train/fold1-gold.json")
text2, gold2 = _json("Train/fold2-text.json"), _json("Train/fold2-gold.json")
text3 = _json("Test/fold3-text.json")
with open(_path("Train/labels.txt"), "r", encoding="utf-8") as f:
    LABELS = [l.split()[0] for l in f if l.strip()]
print(f"Labels ({len(LABELS)}): {LABELS}")
assert len(LABELS) == 12, "labels.txt did not parse into 12 labels — paste me the first lines of the file."

def as_text(v): return v.get("text") if isinstance(v, dict) else v

def parse_entity(e):
    if isinstance(e, (list, tuple)):
        return int(e[0]), int(e[1]), e[2], (e[3] if len(e) > 3 else None)
    if isinstance(e, dict):
        g = lambda *ks: next((e[k] for k in ks if k in e), None)
        return (int(g("start", "start_char", "begin")), int(g("end", "end_char")),
                g("label", "type", "entity"), g("text", "entity_text", "span"))
    raise ValueError(f"Unknown annotation format: {e!r}")

def doc_entities(gold, doc_id):
    v = gold.get(doc_id, [])
    if isinstance(v, dict):
        v = next((x for x in v.values() if isinstance(x, list)), [])
    return [parse_entity(e) for e in v]

# ---- offset convention: is the gold `end` exclusive (python slice) or inclusive? ----
excl = incl = 0
for tx, gd in [(text1, gold1), (text2, gold2)]:
    for did, v in tx.items():
        t = as_text(v)
        for s, e, lab, et in doc_entities(gd, did):
            if et is None: continue
            if t[s:e] == et: excl += 1
            elif t[s:e + 1] == et: incl += 1
print(f"Offset test -> text[s:e]==entity: {excl} | text[s:e+1]==entity: {incl}")
if excl == 0 and incl == 0:
    print("⚠️ Could not verify the convention (no entity-text field matched). Assuming END-EXCLUSIVE.")
END_EXCL = excl >= incl
print("Gold end offset is", "EXCLUSIVE" if END_EXCL else "INCLUSIVE")

# ============================================================
# 2. BIO label space + token labelling / span decoding
# ============================================================
L2I = {"O": 0}
for lab in LABELS:
    L2I["B-" + lab] = len(L2I); L2I["I-" + lab] = len(L2I)
I2L = {v: k for k, v in L2I.items()}
NL = len(L2I)
print(f"Classes: {NL} (O + B/I for each of {len(LABELS)} labels)")

def label_tokens(offsets, ents):
    """ents: [(start, end_exclusive, label)] -> BIO ids per token. Overlapping entities: first (longest) wins."""
    n = len(offsets)
    starts = np.array([o[0] for o in offsets]); ends = np.array([o[1] for o in offsets])
    y = [0] * n; conflicts = 0
    for s, e, lab in sorted(ents, key=lambda x: (x[0], -(x[1] - x[0]))):
        i = int(np.searchsorted(ends, s, side="right"))   # first token ending after s
        first = True
        while i < n and starts[i] < e:
            if ends[i] > starts[i]:
                if y[i] != 0: conflicts += 1
                else:
                    y[i] = L2I[("B-" if first else "I-") + lab]; first = False
            i += 1
    return y, conflicts

def decode_spans(text, offsets, tags):
    """Token tag ids -> [(start, end_exclusive, label)], whitespace trimmed."""
    spans, cur = [], None
    for i, t in enumerate(tags):
        a, b = offsets[i]
        if b <= a: continue
        name = I2L[int(t)]
        if name == "O":
            if cur: spans.append(cur); cur = None
            continue
        pre, lab = name.split("-", 1)
        if pre == "B" or cur is None or cur[2] != lab:
            if cur: spans.append(cur)
            cur = [a, b, lab]
        else:
            cur[1] = b
    if cur: spans.append(cur)
    out = []
    for s, e, lab in spans:
        while s < e and text[s].isspace(): s += 1
        while e > s and text[e - 1].isspace(): e -= 1
        if e > s: out.append((s, e, lab))
    return out

def spans_out(text, offsets, tags):
    return [[int(s), int(e if END_EXCL else e - 1), lab, text[s:e]] for s, e, lab in decode_spans(text, offsets, tags)]

# ============================================================
# 3. Build docs, split by DOCUMENT, windows
# ============================================================
def build_docs(text_d, gold_d, prefix=""):
    docs = []
    for did, v in text_d.items():
        t = as_text(v); enc = tokenizer.encode(t)
        d = {"id": prefix + str(did), "text": t, "ids": enc.ids, "offsets": enc.offsets}
        if gold_d is not None:
            raw = doc_entities(gold_d, did)
            bad = {lab for _, _, lab, _ in raw if lab not in LABELS}
            assert not bad, f"Gold labels not in labels.txt: {bad}"
            d["gold_raw"] = [[s, e, lab] for s, e, lab, _ in raw]
            d["labels"], d["conflicts"] = label_tokens(
                d["offsets"], [(s, e if END_EXCL else e + 1, lab) for s, e, lab, _ in raw])
        docs.append(d)
    return docs

t0 = time.time()
all_docs = build_docs(text1, gold1, "f1:") + build_docs(text2, gold2, "f2:")
test_docs = build_docs(text3, None)
print(f"Tokenized {len(all_docs)} train docs + {len(test_docs)} test docs in {time.time()-t0:.0f}s")

order = list(range(len(all_docs))); random.Random(SEED).shuffle(order)
n_val = max(1, int(VAL_FRACTION * len(all_docs)))
val_docs = [all_docs[i] for i in order[:n_val]]
train_docs = [all_docs[i] for i in order[n_val:]]

def make_windows(n, W=WINDOW, S=STRIDE):
    if n <= W: return [0]
    starts = list(range(0, n - W + 1, S))
    if starts[-1] + W < n: starts.append(n - W)
    return starts

train_windows = [(d["ids"][s:s + WINDOW], d["labels"][s:s + WINDOW])
                 for d in train_docs for s in make_windows(len(d["ids"]))]

tok_tr = sum(len(d["ids"]) for d in train_docs)
print(f"Train docs: {len(train_docs)} ({tok_tr:,} tokens) -> {len(train_windows)} windows of ≤{WINDOW} "
      f"(~{sum(len(x) for x,_ in train_windows)/1e6:.2f}M tokens/epoch)")
print(f"Val docs: {len(val_docs)} | Test docs: {len(test_docs)} | "
      f"docs longer than WINDOW: {sum(len(d['ids'])>WINDOW for d in all_docs)} | max tokens: {max(len(d['ids']) for d in all_docs)}")
print(f"Entity-token conflicts (overlapping gold): {sum(d['conflicts'] for d in all_docs)}")
val_cnt = collections.Counter(l for d in val_docs for _, _, l in d["gold_raw"])
print("Val entity counts:", dict(val_cnt.most_common()))

# class weights (sqrt inverse frequency, clipped)
cnt = np.bincount(np.concatenate([np.array(d["labels"]) for d in train_docs]), minlength=NL).astype(np.float64)
w = np.ones(NL)
if USE_CLASS_WEIGHTS:
    for c in range(1, NL): w[c] = float(np.clip(math.sqrt(cnt[0] / max(cnt[c], 1.0)), 1.0, CLASS_W_CLIP))
CLASS_W = torch.tensor(w, dtype=torch.float32, device=device) if USE_CLASS_WEIGHTS else None
print(f"O tokens: {100*cnt[0]/cnt.sum():.1f}% | class weights (min/max entity): {w[1:].min():.1f}/{w[1:].max():.1f}")

def collate_windows(items):
    L = max(len(x) for x, _ in items)
    X = torch.full((len(items), L), PAD_ID, dtype=torch.long)
    Y = torch.full((len(items), L), -100, dtype=torch.long)
    for i, (x, y) in enumerate(items):
        X[i, :len(x)] = torch.tensor(x); Y[i, :len(y)] = torch.tensor(y)
    return X, Y

# ============================================================
# 4. ORACLE round-trip: gold -> token labels -> spans -> strict score  (the ceiling)
# ============================================================
o_docs = train_docs + val_docs
o_gold = {d["id"]: d["gold_raw"] for d in o_docs}
o_pred = {d["id"]: spans_out(d["text"], d["offsets"], d["labels"]) for d in o_docs}
o_res = strict_scores(o_gold, o_pred, LABELS)
print("\n=== ORACLE (ceiling for any model with this tokenizer + labelling) ===")
print_scores(o_res)
_lost = []
for d in o_docs:
    got = {(s, e, l) for s, e, l, _ in o_pred[d["id"]]}
    for s, e, l in d["gold_raw"]:
        if (s, e, l) not in got and len(_lost) < 5: _lost.append((d["id"], s, e, l, d["text"][s:e if END_EXCL else e + 1]))
print("Examples of gold spans the round-trip cannot reproduce:", _lost)
assert o_res["macro_f1"] > 0.8, "Oracle ceiling too low — offsets/tokenizer alignment is off. Paste this output to me."

# ============================================================
# 5. NER wrapper around the GQA decoder (no change to the LM class)
# ============================================================
class NERModel(nn.Module):
    """Token classifier: GQA decoder backbone (causal) + linear head.
    shift=k: token i is classified from the hidden state at position i+k (k-token lookahead)."""
    def __init__(self, lm, d_model, n_labels, dropout=0.1, shift=0):
        super().__init__()
        self.lm = lm
        self.shift = shift
        self.drop = nn.Dropout(dropout)
        self.head = nn.Linear(d_model, n_labels)
        nn.init.normal_(self.head.weight, std=0.02); nn.init.zeros_(self.head.bias)

    def forward(self, input_ids):
        lm = self.lm
        x = lm.token_embedding(input_ids)
        for layer in lm.layers:
            x, _ = layer(x, lm.cos, lm.sin, past_kv=None)
        out = self.head(self.drop(lm.norm(x)))
        k = self.shift
        if k > 0:   # position i now holds the prediction for token i (made from position i+k)
            out = torch.cat([out[:, k:], out.new_zeros(out.shape[0], k, out.shape[2])], dim=1)
        return out

# ============================================================
# 6. Prediction (windows averaged) + test-file writer
# ============================================================
@torch.no_grad()
def predict_docs(ner, docs, batch=8):
    ner.eval(); out = {}
    for d in docs:
        n = len(d["ids"])
        probs = torch.zeros(n, NL, device=device)
        starts = make_windows(n)
        for i in range(0, len(starts), batch):
            chunk = starts[i:i + batch]
            seqs = [d["ids"][s:s + WINDOW] for s in chunk]
            L = max(len(q) for q in seqs)
            x = torch.full((len(seqs), L), PAD_ID, dtype=torch.long)
            for j, q in enumerate(seqs): x[j, :len(q)] = torch.tensor(q)
            with amp(): lg = ner(x.to(device))
            p = F.softmax(lg.float(), dim=-1)
            k = ner.shift
            for j, (s, q) in enumerate(zip(chunk, seqs)):
                valid = len(q) if (s + len(q) >= n) else len(q) - k   # last k tokens of a window have no lookahead
                probs[s:s + valid] += p[j, :valid]
        out[d["id"]] = spans_out(d["text"], d["offsets"], probs.argmax(-1).tolist())
    return out

def write_test_predictions(ner, name):
    preds = predict_docs(ner, test_docs)
    path = f"{P2_DIR}/prediction_{name}.json"
    with open(path, "w", encoding="utf-8") as f: json.dump(preds, f, ensure_ascii=False)
    print(f"Saved {len(preds)} test docs / {sum(len(v) for v in preds.values())} entities -> {path}")
    return path

# ============================================================
# 7. Trainer (optionally with distillation from a teacher NER model)
# ============================================================
def train_ner(ner, name, teacher=None, alpha=0.0, T=2.0, epochs=10, patience=3,
              backbone_lr=1e-4, head_lr=1e-3, batch_size=8, weight_decay=0.01,
              warmup_frac=0.06, debug=False):
    tw = train_windows[:batch_size * 10] if debug else train_windows
    vdocs = val_docs[:10] if debug else val_docs
    if debug: epochs = 1; name = name + "_debug"
    vgold = {d["id"]: d["gold_raw"] for d in vdocs}
    ner.to(device)
    if teacher is not None:
        teacher.to(device).eval()
        for p in teacher.parameters(): p.requires_grad_(False)

    groups = {}
    for n_, p in ner.named_parameters():
        if p.requires_grad: groups.setdefault((n_.startswith("head."), p.dim() >= 2), []).append(p)
    optimizer = torch.optim.AdamW(
        [{"params": ps, "lr": head_lr if h else backbone_lr, "weight_decay": weight_decay if dec else 0.0}
         for (h, dec), ps in groups.items()], betas=(0.9, 0.98))
    spe = math.ceil(len(tw) / batch_size); total = spe * epochs; warm = max(1, int(warmup_frac * total))
    sched = LambdaLR(optimizer, lambda s: (s + 1) / warm if s < warm else max(0.0, (total - s) / max(1, total - warm)))

    n_train = sum(p.numel() for p in ner.parameters() if p.requires_grad)
    print(f"[{name}] trainable params: {n_train/1e6:.2f}M | windows: {len(tw)} | steps/epoch: {spe} | "
          f"KD: {'alpha=%.2f T=%.1f' % (alpha, T) if teacher is not None and alpha > 0 else 'off'}")
    if torch.cuda.is_available(): torch.cuda.reset_peak_memory_stats()

    best_f1, best_res, bad, history = -1.0, None, 0, []
    best_path = f"{P2_DIR}/{name}_best.pt"; t0 = time.time()

    if teacher is not None: assert teacher.shift == ner.shift, "teacher and student must use the same shift"
    for ep in range(epochs):
        ner.train(); idx = list(range(len(tw))); random.shuffle(idx)
        loss_sum = torch.tensor(0.0, device=device); nb = 0
        bar = tqdm(range(0, len(idx), batch_size), desc=f"{name} ep {ep+1}/{epochs}")
        for i in bar:
            X, Y = collate_windows([tw[j] for j in idx[i:i + batch_size]])
            X, Y = X.to(device), Y.to(device)
            if ner.shift > 0: Y[:, -ner.shift:] = -100        # the last k columns have no lookahead
            with amp(): lg = ner(X)
            lg = lg.float()
            loss = F.cross_entropy(lg.reshape(-1, NL), Y.reshape(-1), weight=CLASS_W, ignore_index=-100)
            if teacher is not None and alpha > 0:
                with torch.no_grad(), amp(): tl = teacher(X)
                m = Y != -100
                kd = F.kl_div(F.log_softmax(lg[m] / T, -1), F.softmax(tl.float()[m] / T, -1),
                              reduction="batchmean") * (T * T)
                loss = (1 - alpha) * loss + alpha * kd
            optimizer.zero_grad(set_to_none=True)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(ner.parameters(), 1.0)
            optimizer.step(); sched.step()
            loss_sum += loss.detach(); nb += 1
            if nb % 20 == 0: bar.set_postfix(loss=f"{(loss_sum / nb).item():.4f}")

        res = strict_scores(vgold, predict_docs(ner, vdocs), LABELS)
        history.append({"epoch": ep + 1, "train_loss": (loss_sum / max(1, nb)).item(), "macro_f1": res["macro_f1"]})
        print(f"epoch {ep+1}: train loss {history[-1]['train_loss']:.4f} | val macro P/R/F1 = "
              f"{res['macro_p']:.4f}/{res['macro_r']:.4f}/{res['macro_f1']:.4f} | {(time.time()-t0)/60:.1f} min")
        if res["macro_f1"] > best_f1:
            best_f1, best_res, bad = res["macro_f1"], res, 0
            torch.save({"model": ner.state_dict(), "epoch": ep + 1, "val_macro_f1": best_f1}, best_path)
        else:
            bad += 1
            if bad >= patience: print(f"Early stop (no improvement for {patience} epochs)."); break

    ner.load_state_dict(torch.load(best_path, map_location=device, weights_only=False)["model"])
    mem = torch.cuda.max_memory_allocated() / 1e9 if torch.cuda.is_available() else 0.0
    print(f"✅ [{name}] best val macro-F1 {best_f1:.4f} | {(time.time()-t0)/60:.1f} min | peak mem {mem:.1f} GB | saved {best_path}")
    return {"res": best_res, "history": history, "best_path": best_path, "minutes": (time.time() - t0) / 60,
            "peak_gb": mem, "trainable_params": n_train}

print("\n✅ Phase-2 toolkit ready.")

fold1-text.json:   0%|          | 0.00/1.11M [00:00<?, ?B/s]

fold1-gold.json:   0%|          | 0.00/461k [00:00<?, ?B/s]

fold2-text.json:   0%|          | 0.00/1.12M [00:00<?, ?B/s]

fold2-gold.json:   0%|          | 0.00/475k [00:00<?, ?B/s]

fold3-text.json:   0%|          | 0.00/1.19M [00:00<?, ?B/s]

labels.txt:   0%|          | 0.00/71.0 [00:00<?, ?B/s]

Labels (12): ['APP', 'RESP', 'A.COUNSEL', 'R.COUNSEL', 'JUDGE', 'WIT', 'AUTH', 'COURT', 'STAT', 'PREC', 'DATE', 'CASENO']
Offset test -> text[s:e]==entity: 8393 | text[s:e+1]==entity: 0
Gold end offset is EXCLUSIVE
Classes: 25 (O + B/I for each of 12 labels)
Tokenized 70 train docs + 35 test docs in 2s
Train docs: 60 (399,913 tokens) -> 530 windows of ≤1024 (~0.54M tokens/epoch)
Val docs: 10 | Test docs: 35 | docs longer than WINDOW: 70 | max tokens: 23710
Entity-token conflicts (overlapping gold): 0
Val entity counts: {'PREC': 212, 'STAT': 171, 'DATE': 157, 'AUTH': 150, 'COURT': 97, 'RESP': 89, 'CASENO': 75, 'APP': 66, 'JUDGE': 32, 'WIT': 23, 'R.COUNSEL': 22, 'A.COUNSEL': 20}
O tokens: 90.8% | class weights (min/max entity): 6.1/10.0

=== ORACLE (ceiling for any model with this tokenizer + labelling) ===
MACRO  P=0.9866  R=0.9866  F1=0.9866  | micro F1=0.9819
label            P       R      F1   gold   pred     tp
DATE         1.000   1.000   1.000   1575   1575   1575
STAT         0.

In [ ]:
for name in ["train_ner", "NERModel", "write_test_predictions", "CustomDecoderLM", "train_windows", "NL", "P2_DIR"]:
    assert name in globals(), f"'{name}' is not defined — run the earlier cells first."

DEBUG_MODE = False      # first run: tiny smoke test. Flip to False for the real run.

TEACHER_CKPT = "/content/drive/MyDrive/CS781_Capstone_Project_Phase_1/gqa_checkpoints/gqa_best.pt"
assert os.path.exists(TEACHER_CKPT), f"Checkpoint not found: {TEACHER_CKPT}"

# ---- load the pretrained Phase-1 model ----
lm = CustomDecoderLM(vocab_size=tokenizer.get_vocab_size(), d_model=768, n_layer=10, n_head=12,
                     n_kv_head=4, max_seq_len=2048, tie_weights=True)
ck = torch.load(TEACHER_CKPT, map_location="cpu", weights_only=False)
lm.load_state_dict(ck["model"] if "model" in ck else ck)
lm.to(device)

# sanity check: the loaded weights should give val loss ≈ 2.2 on Phase-1 val blocks
if "val_dataset" in globals():
    lm.eval(); ls = []
    with torch.no_grad():
        for i in range(0, 32, 4):
            ids = torch.stack([val_dataset[j]["input_ids"] for j in range(i, i + 4)]).to(device).long()
            with amp(): ls.append(lm(ids, labels=ids)["loss"].float().item())
    print(f"Loaded teacher: val loss ≈ {np.mean(ls):.4f} (expect ≈ 2.2)")

# ---- full fine-tuning ----
ner_teacher = NERModel(lm, d_model=768, n_labels=NL, dropout=0.1)
out_t = train_ner(ner_teacher, "ner_teacher_gqa", epochs=10, patience=3,
                  backbone_lr=1e-4, head_lr=1e-3, batch_size=8, debug=DEBUG_MODE)

print("\n=== Validation (strict) — best epoch ===")
print_scores(out_t["res"])
if not DEBUG_MODE:
    write_test_predictions(ner_teacher, "sub1_gqa_full_ft")
else:
    print("\n*** DEBUG_MODE smoke test finished. If it ran cleanly, set DEBUG_MODE=False and re-run. ***")

Loaded teacher: val loss ≈ 2.1824 (expect ≈ 2.2)
[ner_teacher_gqa] trainable params: 87.53M | windows: 530 | steps/epoch: 67 | KD: off


ner_teacher_gqa ep 1/10:   0%|          | 0/67 [00:00<?, ?it/s]

epoch 1: train loss 1.1636 | val macro P/R/F1 = 0.1507/0.3003/0.1982 | 0.1 min


ner_teacher_gqa ep 2/10:   0%|          | 0/67 [00:00<?, ?it/s]

epoch 2: train loss 0.1451 | val macro P/R/F1 = 0.1643/0.3477/0.2204 | 0.3 min


ner_teacher_gqa ep 3/10:   0%|          | 0/67 [00:00<?, ?it/s]

... (7 outputs left out)

ner_teacher_gqa ep 7/10:   0%|          | 0/67 [00:00<?, ?it/s]

epoch 7: train loss 0.0048 | val macro P/R/F1 = 0.2639/0.3307/0.2868 | 1.1 min


ner_teacher_gqa ep 8/10:   0%|          | 0/67 [00:00<?, ?it/s]

epoch 8: train loss 0.0030 | val macro P/R/F1 = 0.2699/0.3267/0.2878 | 1.3 min
Early stop (no improvement for 3 epochs).
✅ [ner_teacher_gqa] best val macro-F1 0.3063 | 1.3 min | peak mem 17.8 GB | saved /content/drive/MyDrive/CS781_Capstone_Project_Phase_2/ner_teacher_gqa_best.pt

=== Validation (strict) — best epoch ===
MACRO  P=0.2725  R=0.3661  F1=0.3063  | micro F1=0.3704
label            P       R      F1   gold   pred     tp
PREC         0.176   0.269   0.213    212    324     57
STAT         0.610   0.696   0.650    171    195    119
DATE         0.867   0.955   0.909    157    173    150
AUTH         0.113   0.107   0.110    150    141     16
COURT        0.483   0.753   0.589     97    151     73
RESP         0.140   0.090   0.110     89     57      8
CASENO       0.143   0.400   0.211     75    210     30
APP          0.079   0.136   0.100     66    114      9
JUDGE        0.457   0.656   0.538     32     46     21
WIT          0.000   0.000   0.000     23     47      0
R.COU

In [ ]:
def make_cw(clip):
    w = np.ones(NL)
    for c in range(1, NL): w[c] = float(np.clip(math.sqrt(cnt[0] / max(cnt[c], 1.0)), 1.0, clip))
    return torch.tensor(w, dtype=torch.float32, device=device)

def fresh_teacher_ner(dropout, shift):
    lm_ = CustomDecoderLM(vocab_size=tokenizer.get_vocab_size(), d_model=768, n_layer=10, n_head=12,
                          n_kv_head=4, max_seq_len=2048, tie_weights=True)
    ck_ = torch.load(TEACHER_CKPT, map_location="cpu", weights_only=False)
    lm_.load_state_dict(ck_["model"] if "model" in ck_ else ck_)
    return NERModel(lm_, d_model=768, n_labels=NL, dropout=dropout, shift=shift).to(device)

VARIANTS = [   # (tag, class-weight clip, backbone_lr, dropout, epochs, seed, shift)
    ("s0_cw10_seed2",      10.0, 1e-4, 0.1, 10, 2, 0),   # old setting, new seed -> noise estimate
    ("s1_cw10",            10.0, 1e-4, 0.1, 10, 1, 1),
    ("s1_noW",             None, 1e-4, 0.1, 10, 1, 1),
    ("s2_noW",             None, 1e-4, 0.1, 10, 1, 2),
    ("s1_noW_lr5e-5_do.3", None, 5e-5, 0.3,  8, 1, 1),
]
orig_cw, rows = CLASS_W, []
for tag, clip, lr, do, ep, seed, sh in VARIANTS:
    random.seed(seed); torch.manual_seed(seed)
    CLASS_W = None if clip is None else make_cw(clip)
    m = fresh_teacher_ner(do, sh)
    o = train_ner(m, f"abl_{tag}", epochs=ep, patience=ep, backbone_lr=lr, head_lr=1e-3, batch_size=8)
    h = [x["macro_f1"] for x in o["history"]]
    lf, _ = lenient_scores({d["id"]: d["gold_raw"] for d in val_docs}, predict_docs(m, val_docs))
    rows.append((tag, max(h), h[-1], o["res"]["macro_p"], o["res"]["macro_r"], lf))
    os.remove(o["best_path"]); del m; torch.cuda.empty_cache()
CLASS_W = orig_cw

print(f"\n{'variant':20s} {'best F1':>8s} {'last F1':>8s} {'P':>7s} {'R':>7s} {'lenient':>8s}")
for r in rows: print(f"{r[0]:20s} {r[1]:8.4f} {r[2]:8.4f} {r[3]:7.4f} {r[4]:7.4f} {r[5]:8.4f}")

[abl_s0_cw10_seed2] trainable params: 87.53M | windows: 530 | steps/epoch: 67 | KD: off


abl_s0_cw10_seed2 ep 1/10:   0%|          | 0/67 [00:00<?, ?it/s]

epoch 1: train loss 1.1735 | val macro P/R/F1 = 0.1630/0.3138/0.2051 | 0.1 min


abl_s0_cw10_seed2 ep 2/10:   0%|          | 0/67 [00:00<?, ?it/s]

epoch 2: train loss 0.1544 | val macro P/R/F1 = 0.1913/0.3159/0.2317 | 0.3 min


abl_s0_cw10_seed2 ep 3/10:   0%|          | 0/67 [00:00<?, ?it/s]

... (87 outputs left out)

abl_s1_noW_lr5e-5_do.3 ep 7/8:   0%|          | 0/67 [00:00<?, ?it/s]

epoch 7: train loss 0.0031 | val macro P/R/F1 = 0.2749/0.3177/0.2893 | 1.0 min


abl_s1_noW_lr5e-5_do.3 ep 8/8:   0%|          | 0/67 [00:00<?, ?it/s]

epoch 8: train loss 0.0017 | val macro P/R/F1 = 0.2752/0.3141/0.2875 | 1.2 min
✅ [abl_s1_noW_lr5e-5_do.3] best val macro-F1 0.2893 | 1.2 min | peak mem 18.5 GB | saved /content/drive/MyDrive/CS781_Capstone_Project_Phase_2/abl_s1_noW_lr5e-5_do.3_best.pt

variant               best F1  last F1       P       R  lenient
s0_cw10_seed2          0.3065   0.2922  0.2745  0.3600   0.5460
s1_cw10                0.2946   0.2946  0.2686  0.3435   0.5352
s1_noW                 0.2829   0.2785  0.2618  0.3234   0.5569
s2_noW                 0.2818   0.2788  0.2691  0.3116   0.5290
s1_noW_lr5e-5_do.3     0.2893   0.2875  0.2749  0.3177   0.5163


In [ ]:
from torch.utils.data import DataLoader, Subset

for name in ["train_ner", "NERModel", "write_test_predictions", "CustomDecoderLM", "train_dataset", "val_dataset", "NL", "P2_DIR"]:
    assert name in globals(), f"'{name}' is not defined — run the earlier cells first."

# ============================================================
# 0. Settings
# ============================================================
DEBUG_MODE = True      # tiny smoke test first; flip to False for the real run
RUN_STAGE1, RUN_STAGE2 = True, True
USE_KD = True              # False -> same-size student trained WITHOUT the teacher (baseline for the report)
S1_ALPHA, S1_T = 0.5, 2.0  # stage 1 loss = (1-a)*CE + a*T^2*KL
S2_ALPHA, S2_T = 0.5, 2.0  # stage 2 (NER) the same idea

STUDENT_CFG = dict(d_model=576, n_layer=7, n_head=12, n_kv_head=4)     # ~43M, about half the teacher
TEACHER_CFG = dict(d_model=768, n_layer=10, n_head=12, n_kv_head=4)
TEACHER_LM_CKPT = "/content/drive/MyDrive/CS781_Capstone_Project_Phase_1/gqa_checkpoints/gqa_best.pt"
TEACHER_NER_CKPT = f"{P2_DIR}/ner_teacher_gqa{'_debug' if DEBUG_MODE else ''}_best.pt"

if DEBUG_MODE:
    S1_STEPS, S1_EVAL_EVERY, S1_WARM, S1_COOL = 20, 10, 5, 5
else:
    S1_STEPS, S1_EVAL_EVERY, S1_WARM, S1_COOL = 4000, 500, 200, 600   # 4000 steps ≈ 131M tokens
S1_BATCH, S1_ACCUM, S1_LR, S1_WD = 4, 4, 5e-4, 0.1

EXP = ("student_kd" if USE_KD else "student_nokd") + ("_debug" if DEBUG_MODE else "")
S1_DIR = f"{P2_DIR}/{EXP}_stage1"; os.makedirs(S1_DIR, exist_ok=True)
S1_BEST, S1_LAST = f"{S1_DIR}/best.pt", f"{S1_DIR}/last.pt"

V = tokenizer.get_vocab_size()
def build_lm(cfg): return CustomDecoderLM(vocab_size=V, max_seq_len=2048, tie_weights=True, **cfg)
def load_sd(path):
    ck = torch.load(path, map_location="cpu", weights_only=False)
    return ck["model"] if "model" in ck else ck

student = build_lm(STUDENT_CFG).to(device)
n_stu = sum(p.numel() for p in student.parameters())
n_tea = sum(p.numel() for p in build_lm(TEACHER_CFG).parameters())
print(f"Student: {n_stu/1e6:.2f}M params | Teacher: {n_tea/1e6:.2f}M | ratio {100*n_stu/n_tea:.0f}%")
assert 0.40 < n_stu / n_tea < 0.60, "Student is not about half the teacher — adjust STUDENT_CFG."

# ============================================================
# 1. STAGE 1 — LM training on Phase-1 blocks (with or without the teacher)
# ============================================================
def blocks_collate(batch): return torch.stack([x["input_ids"] for x in batch])

if RUN_STAGE1:
    teacher_lm = None
    if USE_KD:
        teacher_lm = build_lm(TEACHER_CFG); teacher_lm.load_state_dict(load_sd(TEACHER_LM_CKPT))
        teacher_lm.to(device).eval()
        for p in teacher_lm.parameters(): p.requires_grad_(False)

    train_loader = DataLoader(train_dataset, batch_size=S1_BATCH, shuffle=True, drop_last=True, collate_fn=blocks_collate)
    val_loader = DataLoader(Subset(val_dataset, range(min(len(val_dataset), 200))), batch_size=S1_BATCH,
                            shuffle=False, collate_fn=blocks_collate)

    @torch.no_grad()
    def s1_eval(m):
        m.eval(); tot = torch.tensor(0.0, device=device); n = 0
        for ids in val_loader:
            ids = ids.to(device).long()
            with amp(): tot += m(ids, labels=ids)["loss"].float()
            n += 1
        m.train() if m is student else None
        return (tot / max(1, n)).item()

    if USE_KD:
        tv = s1_eval(teacher_lm); print(f"Teacher val loss on the same blocks: {tv:.4f} (ppl {math.exp(tv):.1f})")

    dec = [p for p in student.parameters() if p.dim() >= 2]; nodec = [p for p in student.parameters() if p.dim() < 2]
    opt = torch.optim.AdamW([{"params": dec, "weight_decay": S1_WD}, {"params": nodec, "weight_decay": 0.0}],
                            lr=S1_LR, betas=(0.9, 0.95))
    def lr_lambda(s):
        if s < S1_WARM: return (s + 1) / S1_WARM
        if s > S1_STEPS - S1_COOL: return max(0.1, (S1_STEPS - s) / max(1, S1_COOL))
        return 1.0
    sch = LambdaLR(opt, lr_lambda)

    step, best_val = 0, float("inf")
    if os.path.exists(S1_LAST) and not DEBUG_MODE:
        ck = torch.load(S1_LAST, map_location=device, weights_only=False)
        student.load_state_dict(ck["model"]); opt.load_state_dict(ck["optimizer"]); sch.load_state_dict(ck["scheduler"])
        step, best_val = ck["step"], ck["best_val"]; print(f"Resumed stage 1 from step {step}")

    def save_s1(path):
        torch.save({"model": student.state_dict(), "optimizer": opt.state_dict(), "scheduler": sch.state_dict(),
                    "step": step, "best_val": best_val}, path)

    def batches_forever():
        while True:
            for b in train_loader: yield b
    gen = batches_forever()

    student.train(); t0, step0 = time.time(), step
    ce_acc = torch.tensor(0.0, device=device); kd_acc = torch.tensor(0.0, device=device); nlog = 0
    tokens_per_step = S1_BATCH * S1_ACCUM * len(train_dataset[0]["input_ids"])
    print(f"\n🚀 Stage 1 ({'KD' if USE_KD else 'no KD'}): {S1_STEPS} steps × {tokens_per_step:,} tokens")
    bar = tqdm(total=S1_STEPS, initial=step)
    while step < S1_STEPS:
        for _ in range(S1_ACCUM):
            ids = next(gen).to(device).long()
            with amp():
                s_logits = student(ids)["logits"][:, :-1]
                if USE_KD:
                    with torch.no_grad(): t_logits = teacher_lm(ids)["logits"][:, :-1]
            s_logits = s_logits.float()
            ce = F.cross_entropy(s_logits.reshape(-1, V), ids[:, 1:].reshape(-1))
            if USE_KD:
                kd = F.kl_div(F.log_softmax(s_logits / S1_T, -1).reshape(-1, V),
                              F.softmax(t_logits.float() / S1_T, -1).reshape(-1, V),
                              reduction="batchmean") * (S1_T ** 2)
                loss = (1 - S1_ALPHA) * ce + S1_ALPHA * kd
                kd_acc += kd.detach() / S1_ACCUM
            else:
                loss = ce
            (loss / S1_ACCUM).backward()
            ce_acc += ce.detach() / S1_ACCUM
        torch.nn.utils.clip_grad_norm_(student.parameters(), 1.0)
        opt.step(); sch.step(); opt.zero_grad(set_to_none=True)
        step += 1; nlog += 1; bar.update(1)

        if step % 25 == 0 or step == S1_STEPS:
            el = time.time() - t0; tok_s = (step - step0) * tokens_per_step / el
            eta = (S1_STEPS - step) * el / max(1, step - step0) / 60
            bar.set_postfix(ce=f"{(ce_acc/nlog).item():.3f}", kd=f"{(kd_acc/nlog).item():.3f}",
                            tok_s=f"{tok_s:,.0f}", eta_min=f"{eta:.0f}")
            ce_acc.zero_(); kd_acc.zero_(); nlog = 0
        if step % S1_EVAL_EVERY == 0 or step == S1_STEPS:
            v = s1_eval(student); student.train()
            print(f"\nstep {step}: student val loss {v:.4f} (ppl {math.exp(v):.1f})")
            if v < best_val: best_val = v; save_s1(S1_BEST)
            save_s1(S1_LAST)
    bar.close()
    student.load_state_dict(torch.load(S1_BEST, map_location=device, weights_only=False)["model"])
    print(f"✅ Stage 1 done. Best student val loss {best_val:.4f} (ppl {math.exp(best_val):.1f})")
    del teacher_lm
    torch.cuda.empty_cache()

# ============================================================
# 2. STAGE 2 — NER fine-tuning of the student (distilled from the fine-tuned teacher)
# ============================================================
if RUN_STAGE2:
    if not RUN_STAGE1:
        student.load_state_dict(load_sd(S1_BEST)); student.to(device)
    ner_student = NERModel(student, d_model=STUDENT_CFG["d_model"], n_labels=NL, dropout=0.1)

    t_ner = None
    if USE_KD:
        assert os.path.exists(TEACHER_NER_CKPT), f"Fine-tune the teacher first (cell 2). Missing: {TEACHER_NER_CKPT}"
        t_lm = build_lm(TEACHER_CFG)
        t_ner = NERModel(t_lm, d_model=TEACHER_CFG["d_model"], n_labels=NL)
        t_ner.load_state_dict(load_sd(TEACHER_NER_CKPT)); t_ner.to(device).eval()

    out_s = train_ner(ner_student, f"ner_{EXP}", teacher=t_ner, alpha=S2_ALPHA if USE_KD else 0.0, T=S2_T,
                      epochs=20, patience=5, backbone_lr=3e-4, head_lr=1e-3, batch_size=8, debug=DEBUG_MODE)
    print("\n=== Student validation (strict) — best epoch ===")
    print_scores(out_s["res"])
    if not DEBUG_MODE:
        write_test_predictions(ner_student, f"sub2_{EXP}")
    else:
        print("\n*** DEBUG_MODE smoke test finished. If it ran cleanly, set DEBUG_MODE=False and re-run. ***")

Student: 43.21M params | Teacher: 87.51M | ratio 49%
Teacher val loss on the same blocks: 2.2067 (ppl 9.1)

🚀 Stage 1 (KD): 4000 steps × 32,768 tokens


  0%|          | 0/4000 [00:00<?, ?it/s]


step 500: student val loss 3.9869 (ppl 53.9)

step 1000: student val loss 3.3787 (ppl 29.3)

step 1500: student val loss 3.0747 (ppl 21.6)

step 2000: student val loss 2.8721 (ppl 17.7)

step 2500: student val loss 2.7889 (ppl 16.3)

step 3000: student val loss 2.7295 (ppl 15.3)

step 3500: student val loss 2.6490 (ppl 14.1)

step 4000: student val loss 2.5347 (ppl 12.6)
✅ Stage 1 done. Best student val loss 2.5347 (ppl 12.6)
[ner_student_kd] trainable params: 43.23M | windows: 530 | steps/epoch: 67 | KD: alpha=0.50 T=2.0


ner_student_kd ep 1/20:   0%|          | 0/67 [00:00<?, ?it/s]

epoch 1: train loss 2.0419 | val macro P/R/F1 = 0.1819/0.2719/0.2101 | 0.1 min


ner_student_kd ep 2/20:   0%|          | 0/67 [00:00<?, ?it/s]

... (29 outputs left out)

ner_student_kd ep 17/20:   0%|          | 0/67 [00:00<?, ?it/s]

epoch 17: train loss 0.0135 | val macro P/R/F1 = 0.2974/0.3617/0.3141 | 2.6 min


ner_student_kd ep 18/20:   0%|          | 0/67 [00:00<?, ?it/s]

epoch 18: train loss 0.0125 | val macro P/R/F1 = 0.2949/0.3576/0.3126 | 2.7 min
Early stop (no improvement for 5 epochs).
✅ [ner_student_kd] best val macro-F1 0.3219 | 2.7 min | peak mem 13.9 GB | saved /content/drive/MyDrive/CS781_Capstone_Project_Phase_2/ner_student_kd_best.pt

=== Student validation (strict) — best epoch ===
MACRO  P=0.3035  R=0.3734  F1=0.3219  | micro F1=0.4052
label            P       R      F1   gold   pred     tp
PREC         0.182   0.250   0.211    212    291     53
STAT         0.568   0.608   0.588    171    183    104
DATE         0.834   0.962   0.893    157    181    151
AUTH         0.185   0.113   0.140    150     92     17
COURT        0.503   0.814   0.622     97    157     79
RESP         0.286   0.090   0.137     89     28      8
CASENO       0.262   0.440   0.328     75    126     33
APP          0.135   0.106   0.119     66     52      7
JUDGE        0.361   0.406   0.382     32     36     13
WIT          0.000   0.000   0.000     23     25      

 LoRA & LoRMA


In [ ]:
for name in ["train_ner", "NERModel", "write_test_predictions", "CustomDecoderLM", "TEACHER_CKPT", "NL", "P2_DIR"]:
    assert name in globals(), f"'{name}' is not defined — run the earlier cells first."

DEBUG_MODE = False    # smoke test first (1 epoch, tiny); set False for the real runs

# ============================================================
# 1. Adapter layers (written from scratch)
# ============================================================
class LoRALinear(nn.Module):
    """y = W0 x + (alpha/r) * B (A x)       A: r x in, B: out x r (B=0 at init)"""
    def __init__(self, base, r, alpha):
        super().__init__()
        self.base, self.r, self.scale = base, r, alpha / r
        self.A = nn.Parameter(torch.empty(r, base.in_features)); nn.init.kaiming_uniform_(self.A, a=math.sqrt(5))
        self.B = nn.Parameter(torch.zeros(base.out_features, r))
    def forward(self, x):
        return self.base(x) + self.scale * F.linear(F.linear(x, self.A), self.B)

class LoRMAPlusLinear(nn.Module):
    """LoRMA+: W = ((alpha/r) B A + I) W0 ; B,A act on the OUTPUT side.  y = W0 x ;  out = y + (alpha/r) B (A y)
       A: r x out, B: out x r (B=0 at init -> exact identity)."""
    def __init__(self, base, r, alpha):
        super().__init__()
        self.base, self.r, self.scale = base, r, alpha / r
        self.A = nn.Parameter(torch.empty(r, base.out_features)); nn.init.kaiming_uniform_(self.A, a=math.sqrt(5))
        self.B = nn.Parameter(torch.zeros(base.out_features, r))
    def forward(self, x):
        y = self.base(x)
        return y + self.scale * F.linear(F.linear(y, self.A), self.B)

def inject_adapters(ner, kind, r, alpha, targets=("q_proj", "k_proj", "v_proj", "out_proj")):
    for p in ner.parameters(): p.requires_grad_(False)           # freeze everything first
    cls = LoRALinear if kind == "lora" else LoRMAPlusLinear
    for layer in ner.lm.layers:
        for nm in targets:
            base = getattr(layer.attn, nm)
            setattr(layer.attn, nm, cls(base, r, alpha).to(base.weight.device))   # new A,B are trainable
    for p in ner.head.parameters(): p.requires_grad_(True)       # the NER head is always trained

@torch.no_grad()
def check_merge(mod):
    """Max |adapter output - merged-weight output| on random input. Should be ~1e-5 or smaller."""
    W0 = mod.base.weight
    x = torch.randn(4, W0.shape[1], device=W0.device)
    ref = mod(x)
    Wm = W0 + mod.scale * (mod.B @ mod.A) if isinstance(mod, LoRALinear) else W0 + (mod.scale * (mod.B @ mod.A)) @ W0
    return (ref - F.linear(x, Wm)).abs().max().item()

def build_best_ner(dropout=0.1):
    """Fresh model from the PRETRAINED Phase-1 checkpoint (never the fine-tuned objects in memory)."""
    lm_ = CustomDecoderLM(vocab_size=tokenizer.get_vocab_size(), d_model=768, n_layer=10, n_head=12,
                          n_kv_head=4, max_seq_len=2048, tie_weights=True)
    ck_ = torch.load(TEACHER_CKPT, map_location="cpu", weights_only=False)
    lm_.load_state_dict(ck_["model"] if "model" in ck_ else ck_)
    return NERModel(lm_, d_model=768, n_labels=NL, dropout=dropout).to(device)

# ============================================================
# 2. Runs: LoRA (r=8) and LoRMA+ (r=10) on the best GQA model, two learning rates each
# ============================================================
RESULTS = []
for kind, r in [("lora", 8), ("lorma", 10)]:
    for lr in [3e-4, 1e-3]:
        random.seed(1); torch.manual_seed(1)
        ner = build_best_ner()

        # identity check: with B=0 the adapted model must equal the original
        ids = torch.randint(0, tokenizer.get_vocab_size(), (1, 64), device=device)
        ner.eval()
        with torch.no_grad(): before = ner(ids)
        inject_adapters(ner, kind, r, alpha=2 * r)
        with torch.no_grad(): after = ner(ids)
        d = (before - after).abs().max().item()
        assert d < 1e-4, f"{kind}: not identity at init (max diff {d})"
        print(f"[{kind}] identity-at-init OK (max diff {d:.1e})")

        name = f"ner_best_{kind}_lr{lr:g}"
        out = train_ner(ner, name, epochs=20, patience=5, backbone_lr=lr, head_lr=1e-3,
                        batch_size=8, debug=DEBUG_MODE)
        mod0 = ner.lm.layers[0].attn.q_proj
        print(f"[{kind}] merge check on layer-0 q_proj: max diff {check_merge(mod0):.1e}")
        if not DEBUG_MODE: write_test_predictions(ner, f"abl_{name}")
        h = [x["macro_f1"] for x in out["history"]]
        RESULTS.append((kind, lr, out["trainable_params"], max(h), out["res"]["macro_p"], out["res"]["macro_r"],
                        out["minutes"], out["peak_gb"]))
        del ner; torch.cuda.empty_cache()

print(f"\n{'method':7s} {'lr':>7s} {'trainable':>10s} {'best F1':>8s} {'P':>7s} {'R':>7s} {'min':>5s} {'GB':>5s}")
for k, lr, tp, f1, p, r_, mi, gb in RESULTS:
    print(f"{k:7s} {lr:7.0e} {tp:10,d} {f1:8.4f} {p:7.4f} {r_:7.4f} {mi:5.1f} {gb:5.1f}")
print(f"\nFull fine-tuning reference: 87.53M trainable, val macro-F1 0.3063")

[lora] identity-at-init OK (max diff 0.0e+00)
[ner_best_lora_lr0.0003] trainable params: 0.43M | windows: 530 | steps/epoch: 67 | KD: off


ner_best_lora_lr0.0003 ep 1/20:   0%|          | 0/67 [00:00<?, ?it/s]

epoch 1: train loss 2.4151 | val macro P/R/F1 = 0.0659/0.1962/0.0953 | 0.1 min


ner_best_lora_lr0.0003 ep 2/20:   0%|          | 0/67 [00:00<?, ?it/s]

epoch 2: train loss 0.3817 | val macro P/R/F1 = 0.1453/0.3081/0.1939 | 0.3 min


ner_best_lora_lr0.0003 ep 3/20:   0%|          | 0/67 [00:00<?, ?it/s]

... (97 outputs left out)

ner_best_lorma_lr0.001 ep 16/20:   0%|          | 0/67 [00:00<?, ?it/s]

epoch 16: train loss 0.0056 | val macro P/R/F1 = 0.2463/0.3390/0.2815 | 2.2 min


ner_best_lorma_lr0.001 ep 17/20:   0%|          | 0/67 [00:00<?, ?it/s]

epoch 17: train loss 0.0041 | val macro P/R/F1 = 0.2421/0.3284/0.2742 | 2.4 min
Early stop (no improvement for 5 epochs).
✅ [ner_best_lorma_lr0.001] best val macro-F1 0.2923 | 2.4 min | peak mem 14.8 GB | saved /content/drive/MyDrive/CS781_Capstone_Project_Phase_2/ner_best_lorma_lr0.001_best.pt
[lorma] merge check on layer-0 q_proj: max diff 7.2e-07
Saved 35 test docs / 5867 entities -> /content/drive/MyDrive/CS781_Capstone_Project_Phase_2/prediction_abl_ner_best_lorma_lr0.001.json

method       lr  trainable  best F1       P       R   min    GB
lora      3e-04    428,825   0.2349  0.1885  0.3309   1.7  16.4
lora      1e-03    428,825   0.2629  0.2179  0.3405   1.7  15.0
lorma     3e-04    428,825   0.2358  0.1877  0.3303   1.6  14.8
lorma     1e-03    428,825   0.2923  0.2457  0.3713   2.4  14.8

Full fine-tuning reference: 87.53M trainable, val macro-F1 0.3063


 Prefix-Tuning

In [ ]:
for name in ["train_ner", "NERModel", "write_test_predictions", "build_best_ner", "train_docs", "NL", "P2_DIR"]:
    assert name in globals(), f"'{name}' is not defined — run the earlier cells first (build_best_ner comes from the LoRA/LoRMA cell)."

DEBUG_MODE = False      # smoke test first; set False for the real runs
PREFIX_LEN = 32

class PrefixNERModel(NERModel):
    """Prefix tuning: frozen backbone + trainable per-layer key/value prefixes (+ the NER head).
    Prefixes are fed to the existing attention through its past_kv argument."""
    def __init__(self, lm, d_model, n_labels, init_ids, dropout=0.1, shift=0):
        super().__init__(lm, d_model, n_labels, dropout, shift)
        for p in self.parameters(): p.requires_grad_(False)          # freeze backbone first
        lm.eval()
        with torch.no_grad():
            dev = next(lm.parameters()).device
            pkv = lm(torch.tensor([init_ids], device=dev))["past_key_values"]   # real K/V of the first P tokens
        self.prefix_k = nn.ParameterList([nn.Parameter(k.detach().clone()) for k, _ in pkv])
        self.prefix_v = nn.ParameterList([nn.Parameter(v.detach().clone()) for _, v in pkv])
        for p in self.head.parameters(): p.requires_grad_(True)        # head is always trained

    def hidden(self, input_ids):
        lm = self.lm; B = input_ids.shape[0]
        x = lm.token_embedding(input_ids)
        for i, layer in enumerate(lm.layers):
            pkv = (self.prefix_k[i].expand(B, -1, -1, -1), self.prefix_v[i].expand(B, -1, -1, -1))
            x, _ = layer(x, lm.cos, lm.sin, past_kv=pkv)
        return lm.norm(x)

    def forward(self, input_ids):
        out = self.head(self.drop(self.hidden(input_ids)))
        k = self.shift
        if k > 0:
            out = torch.cat([out[:, k:], out.new_zeros(out.shape[0], k, out.shape[2])], dim=1)
        return out

@torch.no_grad()
def plain_hidden(lm, ids):
    x = lm.token_embedding(ids)
    for layer in lm.layers: x, _ = layer(x, lm.cos, lm.sin, past_kv=None)
    return lm.norm(x)

PREFIX_RESULTS = []
for lr in [1e-3, 5e-3]:
    random.seed(1); torch.manual_seed(1)
    base = build_best_ner()
    init_ids = train_docs[0]["ids"][:PREFIX_LEN]
    ner = PrefixNERModel(base.lm, d_model=768, n_labels=NL, init_ids=init_ids).to(device)

    # correctness check: prefix path == running the model on [prefix tokens + input], at initialisation
    ner.eval()
    ids = torch.randint(0, tokenizer.get_vocab_size(), (1, 64), device=device)
    with torch.no_grad():
        h_prefix = ner.hidden(ids)
        full = torch.cat([torch.tensor([init_ids], device=device), ids], dim=1)
        h_full = plain_hidden(ner.lm, full)[:, PREFIX_LEN:]
    d = (h_prefix - h_full).abs().max().item()
    assert d < 1e-3, f"prefix path does not match the reference run (max diff {d})"
    print(f"[prefix] init check OK (max diff {d:.1e})")

    name = f"ner_best_prefix_lr{lr:g}"
    out = train_ner(ner, name, epochs=30, patience=6, backbone_lr=lr, head_lr=1e-3,
                    batch_size=8, weight_decay=0.0, debug=DEBUG_MODE)
    if not DEBUG_MODE: write_test_predictions(ner, f"abl_{name}")
    h = [x["macro_f1"] for x in out["history"]]
    PREFIX_RESULTS.append((lr, out["trainable_params"], max(h), out["res"]["macro_p"], out["res"]["macro_r"],
                           out["minutes"], out["peak_gb"]))
    del ner, base; torch.cuda.empty_cache()

print(f"\n{'method':7s} {'lr':>7s} {'trainable':>10s} {'best F1':>8s} {'P':>7s} {'R':>7s} {'min':>5s} {'GB':>5s}")
for lr, tp, f1, p, r_, mi, gb in PREFIX_RESULTS:
    print(f"{'prefix':7s} {lr:7.0e} {tp:10,d} {f1:8.4f} {p:7.4f} {r_:7.4f} {mi:5.1f} {gb:5.1f}")
print("\nReference (val macro-F1): full fine-tuning 0.3063 | LoRMA+ 0.2923 | LoRA 0.2629")

[prefix] init check OK (max diff 4.8e-05)
[ner_best_prefix_lr0.001] trainable params: 0.18M | windows: 530 | steps/epoch: 67 | KD: off


ner_best_prefix_lr0.001 ep 1/30:   0%|          | 0/67 [00:00<?, ?it/s]

epoch 1: train loss 3.2331 | val macro P/R/F1 = 0.0086/0.0966/0.0157 | 0.1 min


ner_best_prefix_lr0.001 ep 2/30:   0%|          | 0/67 [00:00<?, ?it/s]

epoch 2: train loss 0.7988 | val macro P/R/F1 = 0.0480/0.2167/0.0770 | 0.3 min


ner_best_prefix_lr0.001 ep 3/30:   0%|          | 0/67 [00:00<?, ?it/s]

... (95 outputs left out)

ner_best_prefix_lr0.005 ep 21/30:   0%|          | 0/67 [00:00<?, ?it/s]

epoch 21: train loss 0.1345 | val macro P/R/F1 = 0.1509/0.2932/0.1948 | 2.8 min


ner_best_prefix_lr0.005 ep 22/30:   0%|          | 0/67 [00:00<?, ?it/s]

epoch 22: train loss 0.1322 | val macro P/R/F1 = 0.1572/0.3000/0.2031 | 3.0 min
Early stop (no improvement for 6 epochs).
✅ [ner_best_prefix_lr0.005] best val macro-F1 0.2118 | 3.0 min | peak mem 14.7 GB | saved /content/drive/MyDrive/CS781_Capstone_Project_Phase_2/ner_best_prefix_lr0.005_best.pt
Saved 35 test docs / 8128 entities -> /content/drive/MyDrive/CS781_Capstone_Project_Phase_2/prediction_abl_ner_best_prefix_lr0.005.json

method       lr  trainable  best F1       P       R   min    GB
prefix    1e-03    183,065   0.1704  0.1233  0.2928   4.1  14.7
prefix    5e-03    183,065   0.2118  0.1668  0.3064   3.0  14.7

Reference (val macro-F1): full fine-tuning 0.3063 | LoRMA+ 0.2923 | LoRA 0.2629
